In [1]:
from pyspark.sql import SparkSession
import getpass

In [2]:
username = getpass.getuser()

In [3]:
spark = SparkSession. \
builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [4]:
spark

In [5]:
sc = spark.sparkContext

In [6]:
orders_base_rdd = sc.textFile("data/retail_db_orders_part-00000")
customers_base_rdd = sc.textFile("data/retail_db_customers_part-00000")

In [7]:
orders_base_rdd.take(5)

['1,2013-07-25 00:00:00.0,11599,CLOSED',
 '2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT',
 '3,2013-07-25 00:00:00.0,12111,COMPLETE',
 '4,2013-07-25 00:00:00.0,8827,CLOSED',
 '5,2013-07-25 00:00:00.0,11318,COMPLETE']

In [8]:
customers_base_rdd.take(5)

['1,Richard,Hernandez,XXXXXXXXX,XXXXXXXXX,6303 Heather Plaza,Brownsville,TX,78521',
 '2,Mary,Barrett,XXXXXXXXX,XXXXXXXXX,9526 Noble Embers Ridge,Littleton,CO,80126',
 '3,Ann,Smith,XXXXXXXXX,XXXXXXXXX,3422 Blue Pioneer Bend,Caguas,PR,00725',
 '4,Mary,Jones,XXXXXXXXX,XXXXXXXXX,8324 Little Common,San Marcos,CA,92069',
 '5,Robert,Hudson,XXXXXXXXX,XXXXXXXXX,"10 Crystal River Mall ",Caguas,PR,00725']

In [9]:
order_items_base_rdd = sc.textFile("data/retail_db_order_items_part-00000")

In [10]:
order_items_base_rdd.take(5)

['1,1,957,1,299.98,299.98',
 '2,2,1073,1,199.99,199.99',
 '3,2,502,5,250.0,50.0',
 '4,2,403,1,129.99,129.99',
 '5,4,897,2,49.98,24.99']

orders   (orders_base_rdd)
========
order_id, order_date, order_customer_id, order_status
#### 1, 2013-07-25 00:00:00.0, 11599, CLOSED

customers    (customers_base_rdd)
==========
customer_id, customer_fname, customer_lname, customer_email, customer_password, customer_street, customer_city, customer_state, customer_zipcode

1, Richard, Hernandez, XXXXXXXXX, XXXXXXXXX, 6303 Heather Plaza, Brownsville, TX, 78521

order_items  (order_items_base_rdd)
==============
order_item_id, order_id, order_item_product_id, order_item_quantity, order_item
_subtotal, order_item_product_price

1, 1, 957, 1, 299.98, 299.98

In [11]:
orders_base_rdd.count()

68883

In [12]:
order_items_base_rdd.count()

172198

In [13]:
customers_base_rdd.count()

12435

## 1. Find top 10 customers who have spent the most amount

In [14]:
# total amount per order_id from order_items table
order_items_map = order_items_base_rdd.map(lambda x : (x.split(",")[1], float(x.split(",")[4])))

In [15]:
order_items_map.take(10)

[('1', 299.98),
 ('2', 199.99),
 ('2', 250.0),
 ('2', 129.99),
 ('4', 49.98),
 ('4', 299.95),
 ('4', 150.0),
 ('4', 199.92),
 ('5', 299.98),
 ('5', 299.95)]

In [16]:
amount_per_order_id = order_items_map.reduceByKey(lambda x,y: x+y)

In [17]:
# order_id, Sum of order_item _subtotal
amount_per_order_id.take(10)

[('35211', 239.96),
 ('35212', 799.77),
 ('35214', 1319.8500000000001),
 ('35215', 399.90999999999997),
 ('35220', 35.98),
 ('35222', 129.99),
 ('35223', 374.92),
 ('35226', 329.99),
 ('35227', 429.97),
 ('35230', 1249.79)]

In [18]:
orders_map = orders_base_rdd.map(lambda x: (x.split(",")[0], x.split(",")[2]))

In [19]:
# order_id, order_customer_id
orders_map.take(5)

[('1', '11599'), ('2', '256'), ('3', '12111'), ('4', '8827'), ('5', '11318')]

In [20]:
customer_id_spend_join = orders_map.join(amount_per_order_id)

In [21]:
customer_id_spend_join.take(10)

[('4', ('8827', 699.85)),
 ('10', ('5648', 651.9200000000001)),
 ('12', ('1837', 1299.8700000000001)),
 ('16', ('7276', 419.93)),
 ('20', ('9198', 879.8599999999999)),
 ('24', ('11441', 829.97)),
 ('44', ('10500', 399.98)),
 ('50', ('5225', 429.97)),
 ('56', ('10519', 699.89)),
 ('57', ('7073', 637.9))]

In [22]:
# Assuming customer could also have placed multiple orders(order_id)
customer_id_spend_join.map(lambda x: x[1]).reduceByKey(lambda x, y: x+y).sortBy(lambda x: x[1], False).take(10)

[('791', 10524.170000000002),
 ('9371', 9299.03),
 ('8766', 9296.14),
 ('1657', 9223.710000000001),
 ('2641', 9130.92),
 ('1288', 9019.11),
 ('3710', 9019.099999999999),
 ('4249', 8918.85),
 ('5654', 8904.95),
 ('5624', 8761.980000000001)]